In [ ]:
from models.end_to_end.tactic_models.retrieval.model import PremiseRetriever

from tqdm import tqdm
from experiments.end_to_end.common import Context, format_augmented_state, zip_strict
import pickle

In [ ]:

retriever = PremiseRetriever.load(
    '../runs/retriever_novel_premises.ckpt', 'cuda', freeze=True
)

In [ ]:
retriever.load_corpus('../runs/indexed_corpus_novel')

In [ ]:
import glob

traces = glob.glob('../runs/bestfs-novel/*')


In [ ]:
from environments.LeanDojo.get_lean_theorems import _get_theorems_from_files

theorems = _get_theorems_from_files('../data/LeanDojo/data/leandojo_benchmark/novel_premises', 'test', None, None, None, 2000)


In [ ]:
repo, theorems, positions = theorems

In [ ]:
theorems = list(zip_strict([repo] * len(theorems), theorems, positions))

In [ ]:
t_dict = {v[1].full_name: v for v in theorems}

In [ ]:
save_dir = '../runs/bestfs-novel-2/'

In [ ]:
for file in tqdm(traces):
    trace = pickle.load(open(file, 'rb')) 
    repo, thm, pos = t_dict[trace.theorem.full_name]

    path = str(thm.file_path)

    for goal, node in trace.nodes.items():
        if node.visit_count > 0:
            retriever_args = Context(path=path, theorem_full_name=thm.full_name, theorem_pos=pos,
                                       state=goal)
            

            retrieved_premises, _ = retriever.retrieve(
                [goal],
                [retriever_args],
                100,
            )
            
            goal = [format_augmented_state(s, premises, 2300, p_drop=0.0)
                for s, premises in zip_strict([goal], retrieved_premises)][0]
          
            setattr(node, 'data', {'augmented_state': goal})

    pickle.dump(trace, open(save_dir + trace.theorem.full_name, 'wb')) 
        
    


In [ ]:
traces[1].split('/')[-1]

In [ ]:
traces = glob.glob('../runs/end_to_end/reprover-v1-novel-2/2024_05_24/00_31_12/traces/0/*')

i = 12

test_a = pickle.load(open(traces[i], 'rb'))
test_b = pickle.load(open(save_dir + traces[i].split('/')[-1], 'rb'))

for g, node in test_a.nodes.items():
    if g in test_b.nodes and node.visit_count > 0 and test_b.nodes[g].visit_count > 0:
        assert node.data['augmented_state'] == test_b.nodes[g].data['augmented_state']
        
        

In [ ]:
test_b.tree.data['augmented_state']

In [ ]:
files = glob.glob(save_dir + '*')
for file in files:
    trace = pickle.load(open(file, 'rb'))

    count = 0
    for g in trace.nodes.keys():
        if hasattr(trace.nodes[g], 'data'):
            count +=1
        
    print (f'count: {count}, total nodes: {len(trace.nodes)}')
        